 # n_estimators vs ROC-AUC
Cross-validated ROC-AUC over estimator counts.


In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

MODEL_DIR = 'random-forest'
MODEL_NAME = 'Random Forest'

cwd = Path.cwd()
project_root = Path("..").resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qspr_config import (
    DATA_PATH,
    ECFP_RADIUS,
    ECFP_N_BITS,
    N_ESTIMATORS,
    N_ESTIMATORS_GRID,
    TOP_N_BITS,
    RANDOM_SEED,
    TEST_SIZE,
    CV_FOLDS,
    N_JOBS,
    FIG_DPI,
    FIGSIZE_SQUARE,
    FIGSIZE_SQUARE,
)
from qspr_common import (
    load_dataset,
    build_feature_matrix,
    make_binary_target,
    apply_plot_style,
    resolve_output_dir,
)

apply_plot_style()
out_dir = resolve_output_dir(MODEL_DIR)


In [2]:
df = load_dataset(DATA_PATH)
df, X = build_feature_matrix(df, radius=ECFP_RADIUS, n_bits=ECFP_N_BITS)


[22:52:24] WARNING: not removing hydrogen atom without neighbors
[22:52:24] WARNING: not removing hydrogen atom without neighbors
[22:52:24] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:25] WARNING: not removing hydrogen atom without neighbors
[22:52:26] WARNING: not removing hydrogen atom without neighbors
[22:52:26] WARNING: not removing hydrogen atom without neighbors
[22:52:26] WARNING: not removing hydrogen atom without neighbors
[22:52:26] WARNING: not removing hydrogen atom without neighbors
[22:52:26] WARNING: not r

In [3]:
y, cutoff = make_binary_target(df["Solubility"].to_numpy())


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

mean_scores = []
std_scores = []
for n_estimators in N_ESTIMATORS_GRID:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
    )
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=N_JOBS)
    mean_scores.append(scores.mean())
    std_scores.append(scores.std())

fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
ax.errorbar(N_ESTIMATORS_GRID, mean_scores, yerr=std_scores, marker="o", capsize=3)
ax.set_xlabel("n_estimators")
ax.set_ylabel("ROC-AUC")
ax.set_title(f"{MODEL_NAME}: n_estimators vs ROC-AUC")

out_path = out_dir / "n_estimators_vs_roc_auc.png"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
out_path
